# Chapter 3 — Linear Systems and Gauss–Jordan Elimination: SageMath Companion

Symbolic, exact-arithmetic counterpart to `code/python/03_linear_systems.ipynb`. Where the Python notebook used NumPy floats, this notebook uses SageMath's `Matrix(QQ, …)` for exact rational arithmetic, native `.rref()`, `.right_kernel()`, and `.solve_right(…)`, plus symbolic systems with parameters.

**To run this notebook:** open in [CoCalc](https://cocalc.com/) (no install needed), or locally with the SageMath kernel (`sage -n jupyter`).

**Kernel:** SageMath.

**Layout:**

1. Exact arithmetic over ℚ — the coffee-blend problem solved cleanly
2. The same problem traced step by step
3. The three outcomes via Rouché–Capelli (rank test)
4. Parametric solutions — `right_kernel()` and the particular + homogeneous decomposition
5. Symbolic systems — *for which `k` is this consistent?*
6. Vandermonde polynomial fit — exactly, in ℚ
7. Mini-application: balancing a chemical equation
8. **Exercises** for you to solve
9. Solutions

## 1. Exact arithmetic — the coffee blend, solved without rounding

Recall §3.0 of the notes:

```
0.5 x  +  0.4 y  +  0.3 z  =  30
0.3 x  +  0.4 y  +  0.3 z  =  25
0.2 x  +  0.2 y  +  0.4 z  =  20
```

Multiply through by 10 to land in ℚ (exact). Sage will solve it in one line, with a guaranteed-exact answer.

In [ ]:
A = Matrix(QQ, [
    [5, 4, 3],
    [3, 4, 3],
    [2, 2, 4],
])
b = vector(QQ, [300, 250, 200])

x = A.solve_right(b)
show('Solution (Morning, Afternoon, Evening) = ', x)
show('Verification A*x = ', A * x, '  (should be ', b, ')')

## 2. Step-by-step Gauss–Jordan, in ℚ

Sage gives us `.rref()` as a one-liner, but understanding what it does requires walking the steps. We'll do this on the worked example from `notes.md` §3.6:

```
 x +  y +  z =  6
2x +  y + 3z = 14
 x + 4y + 9z = 36
```

Build the augmented matrix and apply elementary row operations one at a time. Sage's `M.add_multiple_of_row(i, j, c)` does `Rᵢ ← Rᵢ + c·Rⱼ`.

In [ ]:
M = matrix(QQ, [
    [1, 1, 1,  6],
    [2, 1, 3, 14],
    [1, 4, 9, 36],
])
show('Start:'); show(M)

# Step 1: zero column 0 below the pivot
M.add_multiple_of_row(1, 0, -2)        # R2 -= 2*R1
show('R2 ← R2 − 2·R1:'); show(M)
M.add_multiple_of_row(2, 0, -1)        # R3 -= R1
show('R3 ← R3 − R1:'); show(M)

# Step 2: pivot in row 1 is -1; scale to 1
M.rescale_row(1, -1)                   # R2 *= -1
show('R2 ← −1·R2:'); show(M)

# Zero column 1 below the new pivot
M.add_multiple_of_row(2, 1, -3)        # R3 -= 3*R2
show('R3 ← R3 − 3·R2:'); show(M)

# Step 3: pivot in row 2 is 11; scale to 1
M.rescale_row(2, 1/11)
show('R3 ← (1/11)·R3:'); show(M)

# Zero above pivots, working upward
M.add_multiple_of_row(1, 2, 1)         # R2 += R3
M.add_multiple_of_row(0, 2, -1)        # R1 -= R3
show('R2 += R3, R1 −= R3:'); show(M)
M.add_multiple_of_row(0, 1, -1)        # R1 -= R2
show('R1 −= R2 (RREF reached):'); show(M)

show('Reading off the answer (x, y, z) = ', M.column(3))
show('Sage built-in .rref() agrees:', matrix(QQ, [
    [1, 1, 1,  6],
    [2, 1, 3, 14],
    [1, 4, 9, 36],
]).rref())

## 3. The three outcomes via Rouché–Capelli

From §3.10:

- `rank(A) < rank([A|b])` → **inconsistent**.
- `rank(A) = rank([A|b]) = n` → **unique**.
- `rank(A) = rank([A|b]) < n` → **infinitely many** (`n - rank(A)` free variables).

Sage gives us `.rank()` directly — no need to compute the full RREF first.

In [ ]:
def classify(A, b):
    A = matrix(QQ, A)
    b = vector(QQ, b)
    aug = A.augment(b)
    rA = A.rank(); rAug = aug.rank(); n = A.ncols()
    if rAug > rA:
        return f'Inconsistent (rank A = {rA}, rank [A|b] = {rAug})'
    if rA == n:
        return f'Unique solution (rank = {rA} = n)'
    return f'Infinitely many; {n - rA} free variable(s) (rank = {rA} < n = {n})'

tests = [
    ([[1, 2], [3, 4]],          [5, 11],   'cross at (1, 2)'),
    ([[1, 1], [2, 2]],          [3,  5],   'parallel lines'),
    ([[1, 1], [2, 2]],          [3,  6],   'same line'),
    ([[1, 2, 3], [2, 4, 6]],    [6, 12],   'rank-1 wide system'),
    ([[1, 1, 1], [2, 1, 3], [1, 4, 9]], [6, 14, 36], 'the worked 3x3'),
]
for A, b, lbl in tests:
    print(f'{lbl:25s}  ->  {classify(A, b)}')

## 4. Parametric solutions — `right_kernel()` and `solve_right`

When a system `Ax = b` has infinitely many solutions, **the full solution set** is

```
x = x_p + (any element of the right kernel of A)
```

where `x_p` is one particular solution and `right_kernel(A)` is the subspace of homogeneous solutions (vectors with `Av = 0`). Sage gives both directly.

We'll redo Example 4 from `worked-examples.md`:

```
 x +  2y -   z =  4
2x +  4y +  3z = 18
 x +  2y +  4z = 14
```

In [ ]:
A = matrix(QQ, [
    [1, 2, -1],
    [2, 4,  3],
    [1, 2,  4],
])
b = vector(QQ, [4, 18, 14])

show('rank(A) =', A.rank(), '  rank([A|b]) =', A.augment(b).rank(),
     '  n =', A.ncols())

x_p = A.solve_right(b)
show('A particular solution x_p = ', x_p)

K = A.right_kernel()
show('Right kernel:', K)
show('Kernel basis:', K.basis())

# Sage's basis is normalized; multiply through to match the integer form from the notes
h = 2 * K.basis()[0]
show('Homogeneous direction (cleared fractions):', h)

show('General solution: x = x_p + t · h, with x_p =', x_p, ', h =', h)

In [ ]:
# Verify symbolically: A * (x_p + t*h) = b for all t
var('t')
x = x_p + t * h
show('A · (x_p + t·h) =', A * x)
show('Equal to b for all t? ', bool(A * x == b))

## 5. Symbolic systems — *for which `k` is the system consistent?*

From Exercise 6 of `exercises.md`:

```
 x +  y +  z = 1
 x + 2y + 4z = k
3x + 5y + 9z = k + 2
```

Here `k` is a symbolic parameter. We want to know exactly which `k` make the system consistent.

**Trick:** scan a few candidate `k` values and check the rank. (Sage's symbolic `echelon_form` will divide by `k - 1` assuming it's nonzero — a classic gotcha. Substituting concrete values bypasses the issue.)

In [ ]:
# Build matrix with symbolic k and check the consistency condition by hand
var('k')
Ak = matrix(SR, [
    [1, 1, 1],
    [1, 2, 4],
    [3, 5, 9],
])
bk = vector(SR, [1, k, k + 2])

# Apply elimination ourselves so the symbolic 'k' is preserved in the bottom row
Maug = Ak.augment(bk).change_ring(SR)
Maug.add_multiple_of_row(1, 0, -1)        # R2 -= R1
Maug.add_multiple_of_row(2, 0, -3)        # R3 -= 3*R1
Maug.add_multiple_of_row(2, 1, -2)        # R3 -= 2*R2
show('After elimination:'); show(Maug)
show('Bottom row last entry =', Maug[2, 3].factor())
show('Consistency requires that entry to be 0  →  k =', solve(Maug[2, 3] == 0, k))

In [ ]:
# At k = 1, find the full solution set
A1 = matrix(QQ, [[1, 1, 1], [1, 2, 4], [3, 5, 9]])
b1 = vector(QQ, [1, 1, 3])
show('rank A =', A1.rank(), ', rank [A|b] =', A1.augment(b1).rank(),
     ', n =', A1.ncols(), '→ infinitely many')
x_p = A1.solve_right(b1)
show('Particular x_p =', x_p)
K = A1.right_kernel()
show('Kernel basis:', K.basis())
show('General solution: x =', x_p, '+ t ·', K.basis()[0])

## 6. Vandermonde polynomial fit — exactly, in ℚ

From the Python notebook §8: fit a quadratic `p(x) = a + b x + c x²` through `(0, 1)`, `(1, 4)`, `(2, 9)`. Solve the Vandermonde system `V · (a, b, c)` = `(1, 4, 9)` exactly.

In [ ]:
xs = [0, 1, 2]
ys = [1, 4, 9]
V = matrix(QQ, [[xi^j for j in range(3)] for xi in xs])
show('Vandermonde V =', V)
coef = V.solve_right(vector(QQ, ys))
show('Coefficients (a, b, c) =', coef)
var('x')
p = sum(coef[j] * x^j for j in range(3))
show('p(x) =', p, '  =', p.factor())
# A larger example: fit a cubic through 4 points
xs2 = [0, 1, 2, 3]; ys2 = [1, 0, 1, 8]
V2 = matrix(QQ, [[xi^j for j in range(4)] for xi in xs2])
c2 = V2.solve_right(vector(QQ, ys2))
show('Cubic through (0,1),(1,0),(2,1),(3,8): coefficients =', c2)
p2 = sum(c2[j] * x^j for j in range(4))
show('p2(x) =', p2.expand())

## 7. Mini-application — balancing a chemical equation

Balance the combustion of propane:

```
a C₃H₈ + b O₂ → c CO₂ + d H₂O
```

Atom-by-atom:

| Element | Constraint |
|---|---|
| C | `3a = c` |
| H | `8a = 2d` |
| O | `2b = 2c + d` |

Move everything to the LHS — a homogeneous system with 3 equations in 4 unknowns. We expect (by §3.10) at least 1 free variable: chemistry doesn't pin down a *scale*, just a *ratio*. Sage will hand us the kernel; we then scale it to the smallest positive integers.

In [ ]:
# Variables: (a, b, c, d)
C = matrix(QQ, [
    [ 3,  0, -1,  0],   # 3a - c = 0
    [ 8,  0,  0, -2],   # 8a - 2d = 0
    [ 0,  2, -2, -1],   # 2b - 2c - d = 0
])
show('Coefficient matrix:'); show(C)
show('Right kernel:', C.right_kernel())
show('Basis:', C.right_kernel().basis())
v = C.right_kernel().basis()[0]
# Scale to clear fractions and to make all entries positive
denom = lcm([qi.denom() for qi in v])
v_int = denom * v
if min(v_int) < 0:
    v_int = -v_int
show('Smallest positive integer solution (a, b, c, d) =', v_int)
print(f'Balanced equation: {v_int[0]} C₃H₈ + {v_int[1]} O₂ → {v_int[2]} CO₂ + {v_int[3]} H₂O')

## 8. Exercises

Try these in the cells below. Solutions are at the bottom — *don't peek*.

**E1.** Solve

```
  x +  y +  z = 0
  x + 2y + 3z = 0
2x + 3y + 4z = 0
```

Find a basis for the right kernel and write the general solution.

**E2.** For

```
 x +    2y +    3z = 5
2x  +  4y  +  6z = 11
```

Use the rank test to decide consistent vs inconsistent without computing the full RREF.

**E3.** Fit a polynomial of the lowest degree passing through `(−1, −1), (0, 0), (1, 1), (2, 8)`. (Hint: Vandermonde.)

**E4.** Symbolically: for which `c` does the system

```
x + y     = 2
x + c·y   = 3
```

have a *unique* solution? When `c` does **not** satisfy that condition, what happens — inconsistent or infinitely many?

**E5.** Balance the equation `a Fe + b O₂ → c Fe₂O₃`. (Two atoms, three unknowns — one free variable expected.)

In [ ]:
# E1: your code here


In [ ]:
# E2: your code here


In [ ]:
# E3: your code here


In [ ]:
# E4: your code here


In [ ]:
# E5: your code here


## 9. Solutions

In [ ]:
# E1 solution
A = matrix(QQ, [[1,1,1],[1,2,3],[2,3,4]])
show('rank =', A.rank(), '→ free variables =', A.ncols() - A.rank())
show('Right kernel basis:', A.right_kernel().basis())
# Multiply by lcm of denominators to clear fractions
v = A.right_kernel().basis()[0]
show('Cleared fractions:', lcm([qi.denom() for qi in v]) * v)

In [ ]:
# E2 solution
A = matrix(QQ, [[1, 2, 3], [2, 4, 6]])
b = vector(QQ, [5, 11])
show('rank A =', A.rank(), ', rank [A|b] =', A.augment(b).rank())
show('-> inconsistent (rank A < rank [A|b])')

In [ ]:
# E3 solution: 4 points -> generically a cubic
xs = [-1, 0, 1, 2]
ys = [-1, 0, 1, 8]
V = matrix(QQ, [[xi^j for j in range(4)] for xi in xs])
coef = V.solve_right(vector(QQ, ys))
show('Coefficients (constant, x, x^2, x^3) =', coef)
var('x')
p = sum(coef[j] * x^j for j in range(4))
show('p(x) =', p.expand())
# Note: the points (-1, -1), (0, 0), (1, 1) all satisfy y = x; the fourth point 
# (2, 8) breaks linearity, forcing a higher-degree term.
show('p(x) factored:', p.factor())

In [ ]:
# E4 solution
var('c')
Ac = matrix(SR, [[1, 1], [1, c]])
show('det A =', Ac.det())
show('Unique solution iff det ≠ 0, i.e. iff c ≠ 1')
# At c = 1: substitute and check
A1 = matrix(QQ, [[1, 1], [1, 1]])
b1 = vector(QQ, [2, 3])
show('At c=1: rank A =', A1.rank(), ', rank [A|b] =', A1.augment(b1).rank(),
     '→ inconsistent (lines parallel and distinct)')
# What if RHS were such that it's redundant? Try b = (2, 2)
b2 = vector(QQ, [2, 2])
show('At c=1, b=(2,2): rank A =', A1.rank(), ', rank [A|b] =', A1.augment(b2).rank(),
     '→ infinitely many (same line twice)')

In [ ]:
# E5 solution: 4 Fe + 3 O2 -> 2 Fe2O3
C = matrix(QQ, [
    [1,  0, -2],   # Fe: a - 2c = 0
    [0,  2, -3],   # O:  2b - 3c = 0
])
show('Right kernel basis:', C.right_kernel().basis())
v = C.right_kernel().basis()[0]
v_int = lcm([qi.denom() for qi in v]) * v
if min(v_int) < 0: v_int = -v_int
show('(a, b, c) =', v_int)
print(f'Balanced: {v_int[0]} Fe + {v_int[1]} O₂ → {v_int[2]} Fe₂O₃')

---

## Where to next

- **Chapter 4 (Sage):** matrix–matrix multiplication, the inverse `A^-1` (when it exists), elementary matrices — each elementary row operation we did by hand here is left-multiplication by an elementary matrix.
- **Chapter 5 (Sage):** `A.column_space()`, `A.right_kernel()` get formal names: **image** and **kernel**. The rank theorem `dim image + dim kernel = n` is exactly the rank test of §3 dressed up as geometry.
- **Chapter 7 (Sage):** when `Ax = b` has *no* solution, `A.solve_right(b)` raises an error — we'll learn to find the **least-squares** approximate solution `A.solve_right(b, check=False)` or `A.augment(b).solve_least_squares(…)`.